<a href="https://colab.research.google.com/github/shivansh2310/Quantitative-Momentum/blob/main/Portfolio_Construction_(Concentration).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### A. The "Diworsification" Problem

If you have a quantitative system that finds the 50 stocks with the absolute best momentum, why would you buy 500 stocks?
Many mutual funds "closet index" by buying hundreds of stocks to avoid deviating too far from the benchmark. Gray and Vogel argue that to capture the momentum premium, you must hold a highly concentrated portfolio (typically the top decile, or about 40 to 50 stocks out of a 500-stock universe). You want pure, undiluted exposure to the behavioral anomaly.

### B. Equal-Weighting vs. Value-Weighting

* Value-Weighting (Market Cap): If you value-weight a momentum portfolio, a mega-cap stock with mediocre momentum might take up 15% of your capital, while a mid-cap stock with massive momentum only gets 1%. You are letting size dictate your risk, rather than your actual alpha signal.

* Equal-Weighting: The core tenet of Quantitative Momentum. If you select 50 stocks, every single stock gets exactly a 2.0% allocation. This maximizes your exposure to the momentum factor across the entire portfolio and forces you to naturally "buy low and sell high" when you rebalance (trimming the ones that grew above 2% and adding to the ones that fell below).

## Target Allocator & Trade Blotter

In [23]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import linregress
import yfinance as yf
import warnings

warnings.filterwarnings('ignore')

In [24]:
universe = [
    'AAPL', 'MSFT', 'AMZN', 'NVDA', 'META', 'TSLA', 'GOOGL',
    'JPM', 'BAC', 'GS', 'MS', 'WFC', 'C',
    'XOM', 'CVX', 'COP', 'EOG', 'SLB',
    'JNJ', 'UNH', 'LLY', 'PFE', 'ABBV',
    'PG', 'KO', 'PEP', 'WMT', 'TGT',
    'CAT', 'DE', 'GE', 'HON', 'MMM'
]
market_proxy = 'SPY'

# Combine for downloading
download_list = universe + [market_proxy]

In [25]:
data = yf.download(download_list, period="2y")['Close']
data = data.dropna(axis=1)

[*********************100%***********************]  34 of 34 completed


In [26]:
spy_close = data[market_proxy]
# Calculate the 200-day Simple Moving Average (SMA)
spy_sma_200 = spy_close.rolling(window=200).mean()

In [27]:
spy_close = data[market_proxy]
latest_date = spy_close.index[-1]

# In a production environment, 'today' is pd.Timestamp.today()
current_date = pd.to_datetime(latest_date)

In [28]:

latest_date = spy_close.index[-1]
current_spy_price = spy_close.loc[latest_date]
current_spy_sma = spy_sma_200.loc[latest_date]

is_bull_regime = current_spy_price > current_spy_sma

stock_data = data[universe] # Exclude SPY for the stock ranking

# 252 days = 12 months, 21 days = 1 month
price_t_minus_1m = stock_data.shift(21)
price_t_minus_12m = stock_data.shift(252)

academic_12m_1m_return = (price_t_minus_1m / price_t_minus_12m) - 1

# We take the top 50% of our universe based on raw momentum
current_momentum = academic_12m_1m_return.loc[latest_date].dropna()
median_momentum = current_momentum.median()
high_momentum_universe = current_momentum[current_momentum >= median_momentum].index.tolist()

In [29]:
print("Calculating Path Smoothness (R-Squared)...")
end_date = latest_date
start_date = data.index[data.index.get_loc(end_date) - 252]
price_paths = data.loc[start_date:end_date, high_momentum_universe]

smoothness_scores = {}
X = np.arange(len(price_paths))

for ticker in high_momentum_universe:
    y = price_paths[ticker].values

    valid_idx = ~np.isnan(y)
    if valid_idx.sum() < 200:
        continue

    X_valid = X[valid_idx]
    y_valid = y[valid_idx]

    slope, intercept, r_value, p_value, std_err = linregress(X_valid, y_valid)

    r_squared = r_value ** 2
    smoothness_scores[ticker] = r_squared


Calculating Path Smoothness (R-Squared)...


In [30]:
quality_df = pd.DataFrame({
    'Raw_12m_1m': current_momentum[high_momentum_universe],
    'Smoothness_R2': pd.Series(smoothness_scores)
})

quality_df['Momentum_Rank'] = quality_df['Raw_12m_1m'].rank(ascending=True)
quality_df['Smoothness_Rank'] = quality_df['Smoothness_R2'].rank(ascending=True)

quality_df['HQ_Score'] = quality_df['Momentum_Rank'] + quality_df['Smoothness_Rank']
quality_df = quality_df.sort_values('HQ_Score', ascending=False)


In [31]:
print("Initializing the Portfolio Allocator...")

# Portfolio Parameters
TOTAL_CAPITAL = 1_000_000  # $1,000,000 AUM
TARGET_POSITIONS = 5       # Concentrated portfolio (Top 5 from our small universe)

# Assuming quality_df from Day 3 is still in memory
# Extract the top N tickers based on the HQ (Frog-in-the-pan) Score
target_tickers = quality_df.head(TARGET_POSITIONS).index.tolist()

# Equal Weight Calculation
# Every stock gets exactly 1/N of the portfolio
target_weight = 1.0 / TARGET_POSITIONS
capital_per_position = TOTAL_CAPITAL * target_weight

print(f"Total AUM: ${TOTAL_CAPITAL:,.2f}")
print(f"Target Weight per Position: {target_weight * 100}% (${capital_per_position:,.2f})")

Initializing the Portfolio Allocator...
Total AUM: $1,000,000.00
Target Weight per Position: 20.0% ($200,000.00)


In [32]:
print("\nGenerating the Trade Blotter...")

# We need the most recent closing prices to calculate exact share amounts
# Assuming 'data' from Day 1/2 is in memory and 'latest_date' is defined
latest_prices = data.loc[latest_date, target_tickers]

trade_blotter = []

for ticker in target_tickers:
    price = latest_prices[ticker]

    # Calculate shares to buy (integer math since we can't buy fractional shares easily in institutional accounts)
    shares_to_buy = int(capital_per_position // price)

    # Actual capital deployed due to rounding
    actual_allocation = shares_to_buy * price
    actual_weight = actual_allocation / TOTAL_CAPITAL

    trade_blotter.append({
        'Ticker': ticker,
        'Action': 'BUY',
        'Shares': shares_to_buy,
        'Price': price,
        'Target Weight': f"{target_weight * 100:.1f}%",
        'Actual Weight': f"{actual_weight * 100:.2f}%",
        'Total Value': actual_allocation
    })

blotter_df = pd.DataFrame(trade_blotter)
blotter_df.set_index('Ticker', inplace=True)


Generating the Trade Blotter...


In [33]:
print("\n" + "="*80)
print(f" OFFICIAL TRADE BLOTTER (Execution Date: {current_date.date()})")
print("="*80)
# Format currency for display
display_blotter = blotter_df.copy()
display_blotter['Price'] = display_blotter['Price'].apply(lambda x: f"${x:,.2f}")
display_blotter['Total Value'] = display_blotter['Total Value'].apply(lambda x: f"${x:,.2f}")
print(display_blotter[['Action', 'Shares', 'Price', 'Target Weight', 'Actual Weight', 'Total Value']])
print("-" * 80)

total_deployed = blotter_df['Total Value'].sum()
cash_drag = TOTAL_CAPITAL - total_deployed
print(f"Total Capital Deployed: ${total_deployed:,.2f}")
print(f"Residual Cash (Friction): ${cash_drag:,.2f}")
print("="*80)

print("\nPORTFOLIO CONSTRUCTION COMPLETE.")
print("The portfolio is perfectly equal-weighted. We are maximizing our exposure to the")
print("behavioral anomaly while rejecting the size bias of market-cap weighting.")


 OFFICIAL TRADE BLOTTER (Execution Date: 2026-06-24)
       Action  Shares    Price Target Weight Actual Weight  Total Value
Ticker                                                                 
CAT       BUY     203  $983.92         20.0%        19.97%  $199,735.76
GOOGL     BUY     579  $345.26         20.0%        19.99%  $199,902.65
JNJ       BUY     833  $240.02         20.0%        19.99%  $199,936.66
C         BUY    1387  $144.17         20.0%        20.00%  $199,963.79
SLB       BUY    4317   $46.33         20.0%        20.00%  $199,985.03
--------------------------------------------------------------------------------
Total Capital Deployed: $999,523.88
Residual Cash (Friction): $476.12

PORTFOLIO CONSTRUCTION COMPLETE.
The portfolio is perfectly equal-weighted. We are maximizing our exposure to the
behavioral anomaly while rejecting the size bias of market-cap weighting.
